# Compile cNMF Evaluation Results into Excel

This notebook collects all evaluation outputs (enrichment, perturbation association, explained variance, etc.) for a given cNMF run and compiles them into a single multi-sheet Excel workbook per (K, threshold) combination.

**Prerequisites**: You should have already run the cNMF evaluation pipeline (see `cNMF_evaluation_pipeline.ipynb`).

## Pipeline Steps

| Step | Description |
|------|-------------|
| **Step 1. Set Parameters** | Define I/O paths, MuData keys, K/threshold grid, and sample labels |
| **Step 2. Load Guide Data** | Load guide assignment AnnData and define helper to attach guide metadata |
| **Step 3. Compile Excel** | For each (K, threshold): load MuData, gather all evaluation DataFrames, build summary sheets, and write to `.xlsx` |

## Expected Output

One Excel file per (K, threshold): `{out_dir}/{run_name}/Eval/{K}_{thresh}/cNMF_{K}_{thresh}.xlsx`

### Output Sheets

| Sheet | Rows | Description | Key Columns |
|-------|------|-------------|-------------|
| **Summary** | 1 per program | Overview of each program with top genes, enrichment highlights, perturbation hit counts, and mean scores per sample | `program_name`, `top10_loaded_genes`, `top30_loaded_genes`, `Automatic Timepoint`, `Total Enriched GO Terms`, `Significant regulators with positive/negative effect {sample}`, `sigfdr0.05_targets_sorted_abslog2fcd_{sample}`, `Top 5 specific regulators (FDR<0.1) {sample}`, `Mean program score {sample}`, `top10_enriched_genesets`, `top10_enriched_go_terms`, `manual_annotation_label` (user fills in) |
| **Program Loadings** | top N genes x programs | Long-format gene loading scores with gene descriptions | `Program`, `Rank`, `Gene`, `Annotation` (NCBI/RefSeq description) |
| **Targets Summary** | 1 per target | Per-target aggregated perturbation stats: expression, cell counts, significant programs, correlated targets | `target_name`, `mean_expression_{sample}`, `# Cells {sample}`, `significant programs {sample}`, `# programs {sample}`, `top 5 specific programs (FDR < 0.1) {sample}`, `top 5 pos/neg correls targets (program log2fc) {sample}` |
| **Sample Association** | 1 per program | Kruskal-Wallis test + Dunn's posthoc for program score differences across samples | `program_name`, `batch_kruskall_wallis_pval`, `{sample}_batch_association_dunn_min_pval`, `{sample}_batch_association_dunn_mean_pval` |
| **Perturbation Association** | 1 per target x program x sample | Full Mann-Whitney U test results (split across multiple sheets if >1M rows) | `target_name`, `program_name`, `specificity_scores`, `beta_hat`, `p-value`, `adj_pval` (Storey Q), `approx_log2FC`, `Sample` |
| **Significant Regulators Only** | subset of above | Same as Perturbation Association, filtered to `adj_pval < 0.05` | Same columns as Perturbation Association |
| **Trait Enrichment** | 1 per term x program | GWAS trait enrichment via Fisher exact test against Open Targets L2G | `Term`, `program_name`, `P-value`, `Adjusted P-value`, `Odds Ratio`, `Combined Score`, `Genes`, `overlap_numerator`, `overlap_denominator` |
| **GO Term Enrichment** | 1 per term x program | GO Biological Process 2023 enrichment via Fisher exact test | Same columns as Trait Enrichment |
| **Geneset Enrichment** | 1 per term x program | Reactome 2022 pathway enrichment via Fisher exact test | Same columns as Trait Enrichment |

In [ ]:
import pandas as pd
import muon as mu
import sys
import os



# Change path to wherever you have repo locally
sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/PerturbNMF/src')


from Stage3_Interpretation.B_Summarization.src import (
    rename_adata_gene_dictionary,
    compile_Program_loading_score_sheet_long, compile_Program_loading_score_sheet_flat, Compile_GO_sheet,
    Compile_Geneset_sheet, Compile_Trait_sheet, Compile_Perturbation_sheet, Compile_Association_sheet, Compile_Explained_variance,
    Compile_Target_Summary_sheet, Compile_Summary_sheet, load_simple_sheets, add_specificity_scores_file, check_program_name_match
)

## Step 1. Set Parameters — Define I/O paths, MuData keys, K/threshold grid, and sample labels


In [ ]:

# ── cNMF output paths ──
out_dir = '/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result'
run_name = '020326_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50'

# ── MuData path (used only if loading a single file outside the loop) ──
mdata_path = '/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/020326_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/adata/cNMF_50_0_2.h5mu'

# ── Save path for intermediate values (specificity scores, etc.) ──
save_path = '/oak/stanford/groups/engreitz/Users/ymo/Project/combined_final_merged_hvg10k/Results/combined_final_merged_hvg10k/Evaluation/150_0_5'

# ── MuData keys ──
prog_key = 'cNMF'                          # modality key for cNMF programs
data_key = 'rna'                           # modality key for RNA expression
categorical_key = 'timepoint'              # obs column for sample/condition labels
guide_targets_key = "guide_targets"        # uns key for guide target names

# ── Compilation settings ──
num_gene = 300                             # number of top genes per program to include in loadings
components = [50]                          # K values to compile
sel_threshs = [0.2]                        # density thresholds to compile
Sample = ['d0','d1','d2','d3']             # sample/condition labels (must match categorical_key values)
perturbation_file_name = 'CRT'             # prefix for perturbation result files (e.g. 'CRT' or 'perturbation_association_results')
non_tagerting_key = ['non-targeting']       # guide target labels used as negative controls
effect_size = 'log2FC'                     # column name for effect size in perturbation results

# ── Column name keys (match your enrichment / perturbation TSV headers) ──
GO_Term_key = "Term"                       # index column in GO enrichment file
GO_Genes_key = "Genes"                     # gene list column in GO enrichment file
Geneset_Term_key = "Term"                  # index column in geneset enrichment file
Geneset_Genes_key = "Genes"                # gene list column in geneset enrichment file
Trait_Term_key = "Term"                    # index column in trait enrichment file
Trait_Genes_key = "Genes"                  # gene list column in trait enrichment file
Perturbation_Sample_key = "Sample"         # column name for sample labels in perturbation results
adjusted_pval_key = "Adjusted P-value"     # column name for adjusted p-value in enrichment files

## Step 2. Load Guide Data — Load AnnData with guide assignments for perturbation analysis


In [ ]:
mdata_guide_path = "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Data/withguide.h5ad"
mdata_guide = mu.read(mdata_guide_path)

In [ ]:
# Helper: copy guide_names, guide_targets, and guide_assignment from guide reference into MuData
def _assign_guide(mdata, mdata_guide):
        mdata['cNMF'].uns['guide_names'] = mdata_guide.uns['guide_names']
        mdata['cNMF'].uns['guide_targets'] = mdata_guide.uns['guide_targets']
        mdata['cNMF'].obsm['guide_assignment'] = mdata_guide.obsm['guide_assignment'].toarray()

## Step 3. Compile Excel — For each (K, threshold), gather all evaluation results and write to .xlsx

Sub-steps per iteration:
* 3a. Load MuData and attach guide metadata
* 3b. Load individual evaluation result DataFrames (loadings, GO, Reactome, trait, perturbation, etc.)
* 3c. Validate that program names are consistent across all DataFrames
* 3d. Build Target Summary (aggregated perturbation effects per target across samples)
* 3e. Build Summary sheet (one row per program with highlights from all evaluations)
* 3f. Write all sheets to a single Excel workbook (see Output Sheets table above for details)


In [ ]:
for sel_thresh in sel_threshs:
    for k in components:  

        output_folder = f"{out_dir}/{run_name}/Interpretation/Summary_table/{k}_{str(sel_thresh).replace('.','_')}"
        os.makedirs(output_folder, exist_ok=True)

        # ── 3a. Load MuData and attach guide metadata ──
        mdata = mu.read('{out_dir}/{run_name}/adata/cNMF_{k}_{sel_thresh}.h5mu'.format(out_dir = out_dir,
                                                                                run_name =run_name,
                                                                                k=k,
                                                                                sel_thresh = str(sel_thresh).replace('.','_')))                                                                      
        _assign_guide(mdata)                                                    
          
        # ── 3b. Load all evaluation result DataFrames ──
        # Returns: program loadings (long & flat), GO, Reactome, trait, perturbation,
        #          categorical association, explained variance, and significant-only perturbation
        df_Program_loading_long, df_Program_loading_flat, df_GO, df_Geneset, \
        df_Trait, df_Perturbation, df_Association, df_Explained_Variance, df_Perturbation_significant_gene_only = load_simple_sheets( 
            mdata, out_dir, run_name, k, sel_thresh, num_gene = num_gene, Sample = Sample, perturbation_file_name = perturbation_file_name,
            GO_Term_key = GO_Term_key, GO_Genes_key = GO_Genes_key,
            Geneset_Term_key = Geneset_Term_key, Geneset_Genes_key = Geneset_Genes_key,
            Trait_Term_key = Trait_Term_key, Trait_Genes_key = Trait_Genes_key,
            Perturbation_Sample_key = Perturbation_Sample_key)

        # ── 3c. Validate program name consistency across all DataFrames ──
        check_program_name_match(mdata, prog_key=prog_key, dataframes =[df_GO, df_Geneset, \
        df_Trait, df_Perturbation, df_Association, df_Explained_Variance, df_Perturbation_significant_gene_only ])

        # ── 3d. Build Target Summary sheet ──
        # Aggregates perturbation effect sizes and significance per target gene across samples
        Perturbation_path_base = f'{out_dir}/{run_name}/Evaluation/{k}_{str(sel_thresh).replace(".", "_")}/{k}_{perturbation_file_name}'
        df_Target_Summary = Compile_Target_Summary_sheet(mdata, Perturbation_path_base, Sample = Sample, categorical_key = categorical_key, 
        prog_key = prog_key, data_key = data_key, guide_targets_key = guide_targets_key, save_path=save_path, effect_size=effect_size)
        
        # ── 3e. Build Summary sheet ──
        # One row per program: top genes, best enrichment terms, explained variance, perturbation hit counts
        df_Summary = Compile_Summary_sheet(mdata, df_GO, df_Geneset, df_Perturbation, df_Program_loading_flat, df_Explained_Variance, Sample = Sample, specicicity_path = save_path,
        categorical_key = categorical_key, non_tagerting_key=non_tagerting_key, effect_size=effect_size, adjusted_pval_key=adjusted_pval_key)

        # ── 3f. Write all sheets to Excel ──
        # Excel row limit is 1,048,575 — large perturbation tables are split across multiple sheets
        print(f'Compiling Excel for K={k}, threshold={sel_thresh}')
        MAX_ROWS = 1048575

        with pd.ExcelWriter(f'{output_folder}/cNMF_{k}_{str(sel_thresh).replace(".", "_")}.xlsx') as writer:
            # Sheet 1: Summary — one row per program with highlights from all evaluations
            df_Summary.to_excel(writer, sheet_name='Summary', index=True)

            # Sheet 2: Program Loadings — long-format: Program, Rank, Gene, Annotation
            df_Program_loading_long.to_excel(writer, sheet_name='Program Loadings', index=True)

            # Sheet 3: Targets Summary — one row per target with expression, cell counts, significant programs
            df_Target_Summary.to_excel(writer, sheet_name='Targets Summary', index=True)

            # Sheet 4: Sample Association — Kruskal-Wallis + Dunn posthoc p-values per program
            if df_Association is not None:
                df_Association.to_excel(writer, sheet_name='Sample Association', index=True)

            # Sheets 5+: Perturbation Association — full U-test results per target x program x sample
            if df_Perturbation is not None:
                # Re-load perturbation results with specificity scores appended
                combined_conditions = []
                for samp in Sample:
                    df_Perturbation_ = add_specificity_scores_file(save_path, Perturbation_path_base, samp)
                    df_Perturbation_[Perturbation_Sample_key] = samp
                    combined_conditions.append(df_Perturbation_)
                df_Perturbation = pd.concat(combined_conditions)
                
                # Split perturbation table across sheets if it exceeds Excel row limit
                for i in range(0, len(df_Perturbation), MAX_ROWS):
                    sheet_num = i // MAX_ROWS + 1
                    df_Perturbation.iloc[i:i+MAX_ROWS].to_excel(writer, sheet_name=f'Perturbation Association {sheet_num}', index=True)

                # Significant regulators only — same columns, filtered to adj_pval < 0.05
                for i in range(0, len(df_Perturbation), MAX_ROWS):
                    sheet_num = i // MAX_ROWS + 1
                    df_Perturbation_significant_gene_only.iloc[i:i+MAX_ROWS].to_excel(writer, sheet_name=f'significant regulators only {sheet_num}', index=True)

            # Trait Enrichment — Fisher exact test: top program genes vs GWAS-linked genes (Open Targets)
            if df_Trait is not None:
                df_Trait.to_excel(writer, sheet_name='Trait Enrichment', index=True)

            # GO Term Enrichment — Fisher exact test: top program genes vs GO Biological Process 2023
            if df_GO is not None:
                df_GO.to_excel(writer, sheet_name='GO Term Enrichment', index=True)

            # Geneset Enrichment — Fisher exact test: top program genes vs Reactome 2022 pathways
            if df_Geneset is not None:
                df_Geneset.to_excel(writer, sheet_name='Geneset Enrichment', index=True)

        print(f'Done. Saved to {output_folder}/cNMF_{k}_{str(sel_thresh).replace(".", "_")}.xlsx')

In [ ]:
df_Perturbation_significant_gene_only